# Part 1 — Data, Universe & Baseline Lead–Lag

**MFE 230GA Project 2 · Conditional Lead-Lag Alpha**

This notebook is the diagnostic run of everything part 1 owns. It is meant to be
executed top to bottom and *read* — every step prints the numbers that would make
the step wrong if they looked different, and the figures it saves are the ones the
report's data section needs.

**Pipeline**

| § | Step | Module |
|---|---|---|
| 1 | Environment and WRDS reachability | `data.wrds_connection` |
| 2 | Offline smoke test on synthetic data | `tests/fake_crsp.py` |
| 3 | Raw pulls: CRSP daily, delistings, FF factors | `data.wrds_fetch` |
| 4 | Delisting merge and the survivorship repair | `data.daily_returns` |
| 5 | Data-quality audit | `data.quality_checks` |
| 6 | SIC → Fama-French 49, point in time | `data.industry_map` |
| 7 | Monthly universe and the attrition table | `data.universe` |
| 8 | Leader / follower assignment | `data.leaders` |
| 9 | Baseline lead–lag test and horizon profile | `baseline` |
| 10 | Save the handoff artefacts for parts 2–5 | — |

**Kernel.** Select *Python (230GA Project 2)*. If it is not listed, create the
environment first (see `README.md`) and register it:

```bash
uv venv .venv.nosync --python 3.12
uv pip install --python .venv.nosync/bin/python -r requirements.txt
.venv.nosync/bin/python -m ipykernel install --user --name 230GA-project2 \
    --display-name "Python (230GA Project 2)"
```

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# The notebook lives in notebooks/; the package lives in src/.  An editable
# install (`-e .` in requirements.txt) makes this unnecessary, but adding the
# path explicitly means the notebook also runs for a teammate who only cloned
# the repo and pip-installed the dependencies.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for extra in (PROJECT_ROOT / "src", PROJECT_ROOT / "tests"):
    if str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4.5), "figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 10})

FIGURES = PROJECT_ROOT / "figures"
RESULTS = PROJECT_ROOT / "results"
FIGURES.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)

print("project root:", PROJECT_ROOT)

## 0. Sample configuration

Everything below is driven by these four choices. They are in one cell so a
robustness run is an edit here and a re-run, not a search through the notebook.

**On the sample period.** 1996 is a defensible start: decimalisation (2001) and
the 1997 tick-size reduction both change the microstructure that a daily
reversal effect lives in, so a sample that begins earlier is really two samples.
Starting in 1996 keeps a pre-decimalisation stretch long enough to use as an
out-of-sample regime for part 5 without letting it dominate.

In [ ]:
START = "1996-01-01"
END = "2024-12-31"

# A short window for developing the notebook — the full pull takes minutes on a
# cold cache.  Set SMOKE = False for the real run.
SMOKE = True
if SMOKE:
    START, END = "2015-01-01", "2019-12-31"

print(f"sample: {START} .. {END}   (SMOKE={SMOKE})")

## 1. Environment and WRDS reachability

`wrds_connection_on()` separates the three ways a WRDS session fails — network,
credentials, session — and prints which one happened. Run it first: every later
cell fails confusingly if this is `False`.

If it returns `False` for credentials, copy `.env.example` to `.env` and fill in
`WRDS_USERNAME` / `WRDS_PASSWORD`. `.env` is git-ignored.

In [ ]:
from lead_lag.data.wrds_connection import PROJECT_ROOT as PKG_ROOT, wrds_connection_on

print("package sees project root:", PKG_ROOT)
print(".env present:", (PKG_ROOT / ".env").exists())

WRDS_UP = wrds_connection_on()
print("WRDS reachable:", WRDS_UP)

## 2. Offline smoke test

Before touching real data, run the whole pipeline on a synthetic panel whose
answer is known. `fake_crsp.make_panel` builds followers whose return is exactly
`0.30 × leader_return[t-1]` plus independent noise, so the baseline test has a
right answer and any off-by-one in the lagging shows up immediately.

This cell is the reason the notebook is runnable on a plane, and it is the first
thing to re-run after changing anything in `src/`.

In [ ]:
from fake_crsp import make_delist, make_factors, make_panel
from conftest import MINI_SICCODES

from lead_lag.baseline import baseline_test, horizon_profile
from lead_lag.data.daily_returns import adjusted_daily_returns
from lead_lag.data.industry_map import attach_industry
from lead_lag.data.leaders import LeaderRule, assign_roles, follower_panel
from lead_lag.data.universe import UniverseRules, build_universe
from lead_lag.data.wrds_fetch import CRSPDaily, CRSPDelist

_raw = make_panel()
_daily = CRSPDaily.from_raw(_raw)
_events = CRSPDelist.from_raw(make_delist(_raw))
_returns, _rep = adjusted_daily_returns(_daily, _events)
_panel = attach_industry(_returns.frame, MINI_SICCODES)
_universe, _ = build_universe(_panel, UniverseRules(min_obs=20))
_roles = assign_roles(_universe, LeaderRule(min_followers=2))
_fp = follower_panel(_panel, _roles)

_r1 = baseline_test(_fp, lag=1, market_col=None)
_r2 = baseline_test(_fp, lag=2, market_col=None)

print(f"planted beta            0.300")
print(f"recovered at lag 1      pooled {_r1.beta_pooled:6.3f} (t={_r1.t_pooled:5.1f})   "
      f"FM {_r1.beta_fm:6.3f} (t={_r1.t_fm:5.1f})")
print(f"placebo  at lag 2       pooled {_r2.beta_pooled:6.3f} (t={_r2.t_pooled:5.1f})   "
      f"FM {_r2.beta_fm:6.3f} (t={_r2.t_fm:5.1f})")

assert abs(_r1.beta_pooled - 0.30) < 0.05, "lag-1 estimate is off — check build_lags"
assert abs(_r2.beta_pooled) < 0.05, "lag-2 placebo fired — the shifting is wrong"
print("\nsmoke test OK")

## 3. Raw pulls

Three tables. The daily file is cached one calendar year per parquet, so a cold
run costs minutes and every later run costs seconds. The first run prints a line
per year — that is the pull, not a hang.

**Expected magnitudes.** Roughly 1.0–1.7 million daily rows per year, falling
over the sample as the number of listed US common stocks shrinks from ~7,000 to
~4,000. A year that comes back much smaller than its neighbours is a truncated
pull, not a quiet market.

In [ ]:
from lead_lag.data.wrds_fetch import (
    load_or_fetch_crsp_daily,
    load_or_fetch_crsp_delist,
    load_or_fetch_factors_daily,
    open_wrds,
)

session = open_wrds() if WRDS_UP else None
try:
    daily = load_or_fetch_crsp_daily(session, START, END)
    delist = load_or_fetch_crsp_delist(session, START, END)
    factors = load_or_fetch_factors_daily(session, START, END)
finally:
    if session is not None:
        session.close()

print(f"\ndaily   {len(daily):>12,} rows   {daily.frame['permno'].nunique():>6,} stocks")
print(f"delist  {len(delist):>12,} events")
print(f"factors {len(factors):>12,} days")
daily.frame.head()

In [ ]:
# Rows and distinct stocks per year — the shape check described above.
by_year = daily.frame.assign(year=daily.frame["date"].dt.year).groupby("year").agg(
    rows=("permno", "size"), stocks=("permno", "nunique"), days=("date", "nunique")
)
display(by_year)

fig, ax = plt.subplots()
ax.plot(by_year.index, by_year["stocks"], marker="o")
ax.set_title("Distinct common stocks per year (CRSP, shrcd 10/11, NYSE/AMEX/NASDAQ)")
ax.set_xlabel("year"); ax.set_ylabel("stocks")
fig.tight_layout(); fig.savefig(FIGURES / "p1_stocks_per_year.png"); plt.show()

## 4. Delisting merge and the survivorship repair

The table below is the one to read carefully. `appended as delist-only row`
counts events whose day had no daily row — the stock's series would simply have
ended without them, which is exactly the survivorship bias that would flatter a
strategy that is long small followers.

`dlret repaired (Shumway)` counts rows where a performance-related delisting had
no delisting return and the literature's substitute (−30% NYSE/AMEX, −55%
NASDAQ) was used. If that count is more than a percent or so of events, say so
in the report and run part 5's `repair=False` variant.

In [ ]:
returns, report = adjusted_daily_returns(daily, delist)

display(report.to_frame())
print("\nrows by source:")
display(returns.frame["source"].value_counts().to_frame("rows"))

share_repaired = report.n_repaired / max(report.n_perf_events, 1)
print(f"\nrepaired as a share of performance-related events: {share_repaired:.1%}")

## 5. Data-quality audit

Four checks: duplicated `(date, permno)` keys (must be zero — every later join
depends on it), stocks whose returns are mostly missing, single-day returns
beyond ±100%, and days with an implausibly thin cross-section.

None of these except the first are errors. They are the numbers the cleaning
section of the report quotes, and the extreme-return count is the input to the
winsorisation decision part 5 has to make.

In [ ]:
from lead_lag.data.quality_checks import blocking_audit, daily_panel_audit

audit_report = daily_panel_audit().run(returns)
print("issues by severity:")
display(audit_report.summary().to_frame("count"))

issues = audit_report.to_frame()
if len(issues):
    display(issues.groupby("issue_type").size().to_frame("findings"))
    display(issues.head(10))

# The blocking audit must be clean — this raises if it is not.
blocking_audit().assert_clean(returns)
print("\nblocking checks clean (no duplicate keys)")

issues.to_csv(RESULTS / "p1_quality_report.csv", index=False)

## 6. SIC → Fama-French 49

The SIC code comes from the CRSP **name row valid on the observation date**, not
the header code, so the assignment is point-in-time. See the module docstring for
why that distinction decides whether the whole study is lookahead-free.

The diagnostic to read: the share of rows landing in industry 49 (`Other`).
Fama and French leave some SIC codes unassigned deliberately, and CRSP's unknown
code (0) joins them. A few percent is normal; a large share means the download
failed or the parse is wrong.

In [ ]:
from lead_lag.data.industry_map import OTHER_INDUSTRY, load_siccodes49

siccodes = load_siccodes49()
print(f"{len(siccodes)} SIC ranges across {siccodes['ff49'].nunique()} industries")

panel = attach_industry(returns.frame, siccodes)

other_share = (panel["ff49"] == OTHER_INDUSTRY).mean()
print(f"rows in industry {OTHER_INDUSTRY} ('Other'): {other_share:.2%}")

counts = (
    panel.groupby(["ff49", "ff49_name"])["permno"].nunique()
    .sort_values(ascending=False).to_frame("stocks")
)
display(counts.head(15))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
top = counts.reset_index().sort_values("stocks")
ax.barh(top["ff49_name"], top["stocks"])
ax.set_title("Distinct stocks per Fama-French 49 industry (full sample)")
ax.set_xlabel("stocks")
fig.tight_layout(); fig.savefig(FIGURES / "p1_industry_sizes.png"); plt.show()

## 7. The monthly universe

Five screens, each decided from the twelve months **ending before** the month the
stock may be traded in. The attrition table is cumulative: each column is how
many stocks are left, not how many that screen touched.

Read the price screen's line first. It is the filter most likely to be driving
any short-horizon reversal result — below a few dollars the bid-ask bounce
produces mechanical negative autocorrelation that is indistinguishable from the
effect the project is looking for.

In [ ]:
rules = UniverseRules(
    formation_months=12,
    min_price=5.0,
    min_nyse_size_pct=0.20,
    min_dollar_vol_pct=0.20,
    min_obs=120,
    min_industry_size=5,
    exclude_other=True,
)
print(rules)

universe, attrition = build_universe(panel, rules)
print(f"\n{len(universe):,} (month, permno) decisions; "
      f"{universe.frame['eligible'].sum():,} eligible")
display(attrition.head())
display(attrition.describe().loc[["mean", "min", "max"]].astype(int))

In [ ]:
fig, ax = plt.subplots()
for col in attrition.columns:
    ax.plot(attrition.index, attrition[col], label=col, lw=1.4)
ax.set_title("Universe attrition — stocks surviving each screen, cumulative")
ax.set_xlabel("month"); ax.set_ylabel("stocks"); ax.legend(fontsize=8, ncol=2)
fig.tight_layout(); fig.savefig(FIGURES / "p1_universe_attrition.png"); plt.show()

# How much each screen costs on its own, averaged over months.
solo = pd.Series({
    col.removeprefix("pass_"): 1 - universe.frame[col].mean()
    for col in ("pass_obs", "pass_price", "pass_size", "pass_liquidity", "pass_industry")
}, name="share failing (standalone)")
display(solo.sort_values(ascending=False).to_frame())

## 8. Leaders and followers

Default rule: the largest formation-window market cap in each industry-month is
the leader, everyone else eligible is a follower.

**The diagnostic that matters is leader turnover.** Real industry leaders do not
change month to month. A high rate means the size ranking is being decided by
noise — usually two firms of nearly equal size trading places — and the lead–lag
coefficient would then be measuring that noise as much as any information flow.
Single-digit percent per month is healthy.

In [ ]:
from lead_lag.data.leaders import leader_returns, leader_turnover

rule = LeaderRule(by="mktcap", n_leaders=1, follower_max_size_pct=1.0, min_followers=4)
roles = assign_roles(universe, rule)

print(roles.frame["role"].value_counts().to_frame("rows"))
print(f"\nindustry-months with a leader: "
      f"{roles.frame.groupby(['month', 'ff49']).ngroups:,}")

turnover = leader_turnover(roles)
print(f"leader changes month to month: {turnover.attrs['overall_rate']:.1%}")

by_industry = (
    turnover.groupby("ff49")["changed"].mean().sort_values(ascending=False)
    .to_frame("turnover rate")
)
display(by_industry.head(10))

In [ ]:
# Who the leaders actually are, most recent month — a sanity check that costs
# nothing and catches a mis-specified universe instantly.
last_month = roles.frame["month"].max()
recent = roles.frame.loc[
    (roles.frame["month"] == last_month) & (roles.frame["role"] == "leader")
].merge(
    universe.frame[["month", "permno", "mktcap"]], on=["month", "permno"], how="left"
).merge(
    counts.reset_index()[["ff49", "ff49_name"]], on="ff49", how="left"
)
recent["mktcap_bn"] = recent["mktcap"] / 1e9
display(
    recent[["ff49", "ff49_name", "permno", "mktcap_bn", "n_in_industry"]]
    .sort_values("mktcap_bn", ascending=False).head(20)
)
print(f"leaders as of {last_month.date()}")

## 9. Baseline lead–lag test

The unconditional version: does yesterday's leader return predict today's
follower return, controlling for the follower's own lagged return and the
contemporaneous market?

Two estimators side by side. **Pooled OLS clustered by date** and
**Fama-MacBeth** with Newey-West errors. They answer the same question and fail
differently; the report should lead with Fama-MacBeth, which is the convention
here, and quote the pooled number as agreement.

A t-statistic in the hundreds would mean the clustering is not working — with
thousands of followers sharing each day, treating them as independent
understates the standard error by more than an order of magnitude.

In [ ]:
fp = follower_panel(panel, roles)
fp = fp.merge(factors.frame[["date", "mktrf"]], on="date", how="left")
print(f"regression panel: {len(fp):,} follower-days, "
      f"{fp['permno'].nunique():,} stocks, {fp['date'].nunique():,} days")

result = baseline_test(fp, lag=1, market_col="mktrf")
display(pd.Series(result.as_row()).to_frame("lag 1"))

In [ ]:
profile = horizon_profile(fp, lags=range(1, 11), market_col="mktrf", verbose=True)
display(profile.round(5))
profile.to_csv(RESULTS / "p1_horizon_profile.csv")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.axhline(0, color="black", lw=0.8)
ax1.plot(profile.index, profile["beta_fm"], marker="o", label="Fama-MacBeth")
ax1.plot(profile.index, profile["beta_pooled"], marker="s", ls="--", label="pooled OLS")
ax1.set_title("Lead-lag coefficient by horizon")
ax1.set_xlabel("lag k (trading days)"); ax1.set_ylabel(r"$b(k)$"); ax1.legend()

ax2.axhline(0, color="black", lw=0.8)
ax2.axhline(2, color="grey", lw=0.8, ls=":"); ax2.axhline(-2, color="grey", lw=0.8, ls=":")
ax2.plot(profile.index, profile["t_fm"], marker="o", label="Fama-MacBeth")
ax2.plot(profile.index, profile["t_pooled"], marker="s", ls="--", label="pooled, clustered")
ax2.set_title("t-statistics"); ax2.set_xlabel("lag k (trading days)"); ax2.legend()

fig.tight_layout(); fig.savefig(FIGURES / "p1_horizon_profile.png"); plt.show()

### The non-parametric version

A coefficient assumes linearity and is sensitive to the tails. Sorting
industry-days into quintiles by the leader's lagged return and averaging the
followers' returns assumes neither.

If the regression says `b > 0` but the bins are flat except the extreme one, the
effect lives in a handful of large leader moves rather than in a general
relationship — which changes everything about whether the eventual strategy is
tradeable, and belongs in the report either way.

In [ ]:
from lead_lag.baseline import quintile_sort

bins = quintile_sort(fp, lag=1, n_bins=5)
display(bins.round(5))
print(bins.attrs.get("note", ""))

body = bins.loc[bins.index > 0]
fig, ax = plt.subplots()
ax.bar(body.index.astype(str), body["mean_ret"] * 1e4)
ax.axhline(0, color="black", lw=0.8)
ax.set_title("Mean next-day follower return by quintile of the leader's lagged return")
ax.set_xlabel("quintile of lagged leader return (1 = lowest)")
ax.set_ylabel("mean follower return (bps)")
fig.tight_layout(); fig.savefig(FIGURES / "p1_quintile_sort.png"); plt.show()

In [ ]:
# Stability: is the mean coefficient a persistent effect or one good year?
from lead_lag.baseline import build_lags, fama_macbeth

_, _, daily_coefs = fama_macbeth(build_lags(fp, lag=1))
annual = daily_coefs.groupby(daily_coefs.index.year).mean()

fig, ax = plt.subplots()
ax.axhline(0, color="black", lw=0.8)
ax.bar(annual.index, annual.values)
ax.set_title("Mean daily lead-lag coefficient by year (Fama-MacBeth first stage)")
ax.set_xlabel("year"); ax.set_ylabel(r"mean $b$")
fig.tight_layout(); fig.savefig(FIGURES / "p1_coefficient_by_year.png"); plt.show()

display(annual.round(4).to_frame("mean b"))

## 10. Handoff artefacts

What parts 2–5 consume, written once so nobody re-derives the universe.

| File | Who needs it | What it is |
|---|---|---|
| `results/p1_universe.parquet` | 4 | monthly eligibility + formation statistics |
| `results/p1_roles.parquet` | 2, 3 | leader / follower per industry-month |
| `results/p1_industry_returns.parquet` | 2 | value-weighted industry returns, lagged weights |
| `results/p1_follower_panel.parquet` | 2, 3 | the regression panel, leader return attached |
| `results/p1_horizon_profile.csv` | 3, 5 | the unconditional numbers to beat |
| `results/p1_quality_report.csv` | 5 | the cleaning appendix |

**One warning for part 2**, repeated from `shocks/__init__.py` because it is the
easiest thing in the project to get wrong: the industry return written here
*includes* the leader. Decomposing the leader's return against it puts the leader
on both sides of the regression and shrinks the leader-specific residual toward
zero by construction. Rebuild it ex-leader before using it that way.

In [ ]:
from lead_lag.data.industry_map import industry_returns

ind_ret = industry_returns(panel)
print(f"industry returns: {len(ind_ret):,} (date, industry) observations")

universe.frame.to_parquet(RESULTS / "p1_universe.parquet", index=False)
roles.frame.to_parquet(RESULTS / "p1_roles.parquet", index=False)
ind_ret.to_parquet(RESULTS / "p1_industry_returns.parquet", index=False)
fp.to_parquet(RESULTS / "p1_follower_panel.parquet", index=False)

for f in sorted(RESULTS.iterdir()):
    print(f"{f.name:<34} {f.stat().st_size / 1e6:8.2f} MB")

## What to write up from this notebook

- **Sample and sources.** CRSP daily (`crsp.dsf` + `crsp.dsenames`, point-in-time
  name rows), CRSP delistings, Fama-French daily factors. Period, stock counts,
  and the per-year figure from §3.
- **Cleaning.** The delisting table from §4 — in particular the orphan count and
  the Shumway repair share — plus the quality summary from §5.
- **Universe.** The five screens with their thresholds, the attrition figure from
  §7, and the standalone cost of each screen.
- **Leaders.** The rule, the turnover rate from §8, and the recent-leader table as
  a face-validity check.
- **Baseline.** The horizon profile and quintile figures, both estimators' numbers,
  and the by-year stability chart. If the baseline is flat, that is the finding
  the executive summary has to lead with.

**Robustness handles already wired**, for part 5: `UniverseRules(min_price=...)`,
`UniverseRules(min_dollar_vol_pct=...)`, `LeaderRule(by="med_dollar_vol")`,
`LeaderRule(n_leaders=3)`, `adjusted_daily_returns(..., repair=False)`, and
dropping rows where `DailyReturns.repaired` is True.